# Transformer-based Tweet Sentiment Classification

مبانی پردازش زبان و گفتار — پروژه پایانی

This notebook is the runnable, end-to-end demo of the project in `src/` and `scripts/`. It loads the raw tweet corpus, applies the cleaning pipeline, fine-tunes a transformer classifier, evaluates it, and inspects a handful of misclassified examples.

All the actual logic lives in the `src/` package — this notebook only wires it together, so every step here can also be run head-lessly through `scripts/run_experiment.py`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import ExperimentConfig
from src.data import apply_split, build_or_load_splits, load_raw_corpus, LABEL_NAMES
from src.preprocessing import clean_corpus
from src.trainer import run_training, evaluate_on_texts
from src.error_analysis import build_error_table, confusion_summary


## 2. Load the raw corpus and inspect a few quirks

The raw text file has some artifacts worth handling explicitly: literal `\uXXXX` escape sequences instead of real characters, stray wrapping quotes, and escaped internal quotes.

In [ ]:
texts, labels = load_raw_corpus(ROOT / "data" / "train_text.txt", ROOT / "data" / "train_labels.txt")
print(f"{len(texts):,} tweets loaded")

from collections import Counter
label_counts = Counter(labels)
for label_id, name in enumerate(LABEL_NAMES):
    print(f"  {name:>8}: {label_counts[label_id]:>6,}  ({label_counts[label_id] / len(labels):.1%})")

print("\nExample of a raw tweet with escaped unicode:")
print(repr(texts[13]))


## 3. Fixed train / dev / test split

Computed once and cached to `data/splits.json`. Every experiment — this notebook included — loads the same indices, so hyperparameter and model choices are always compared on an identical dev set, and the test set stays untouched until the very end.

In [ ]:
splits = build_or_load_splits(
    n_samples=len(texts),
    labels=labels,
    split_path=ROOT / "data" / "splits.json",
)
print(f"train={len(splits.train):,}  dev={len(splits.dev):,}  test={len(splits.test):,}")


## 4. Preprocessing

The cleaning pipeline is configurable (see `src/preprocessing.py`), so preprocessing strategy can itself be one of the experimental variables. Here we apply the baseline configuration used by `configs/bert_baseline.yaml`.

In [ ]:
config = ExperimentConfig.from_yaml(ROOT / "configs" / "bert_baseline.yaml")

cleaned_texts = clean_corpus(texts, config.preprocessing)
train_texts, train_labels = apply_split(cleaned_texts, labels, splits.train)
dev_texts, dev_labels = apply_split(cleaned_texts, labels, splits.dev)
test_texts, test_labels = apply_split(cleaned_texts, labels, splits.test)

print("Before:", repr(texts[13]))
print("After :", repr(cleaned_texts[13]))


## 5. Fine-tune

`run_training` handles batching, the linear warmup/decay schedule, per-epoch dev evaluation, and checkpointing the best epoch by dev F1. Swap `config` for any file in `configs/` to reproduce a different experiment (model, optimizer, learning rate, max sequence length, or fine-tuning strategy).

> Requires a GPU-backed runtime and internet access to download pretrained weights the first time each model is used.

In [ ]:
outcome = run_training(config, train_texts, train_labels, dev_texts, dev_labels)
print(f"Best dev F1: {outcome['best_dev_f1']:.4f}")


## 6. Evaluate on the development set

In [ ]:
dev_metrics, dev_report, dev_true, dev_pred = evaluate_on_texts(
    outcome["model"], outcome["tokenizer"], config, dev_texts, dev_labels
)
print(dev_metrics)
print()
print(dev_report)


## 7. Error analysis

A quick look at where the model disagrees with the gold label, plus a confusion matrix to see which classes get confused with each other most often.

In [ ]:
confusion_summary(dev_true, dev_pred)


In [ ]:
build_error_table(dev_texts, dev_true, dev_pred, max_rows=20)


## 8. Final test-set evaluation

Run **once**, after all model/hyperparameter decisions have already been made using the dev set above.

In [ ]:
test_metrics, test_report, test_true, test_pred = evaluate_on_texts(
    outcome["model"], outcome["tokenizer"], config, test_texts, test_labels
)
print(test_metrics)
print()
print(test_report)
